In [ ]:
import pandas as pd
import glob

from openai import OpenAI

from tenacity import (
                        retry,
                        stop_after_attempt,
                        wait_random_exponential
)

from tqdm import tqdm

In [ ]:
def generate_prompt_table_prediction(table, table_name):
    table = table.sample(30, random_state=42)
    system_msg = f"""
            Given a table name, its respective columns names and some column values, describe the semantics of the given table.
            
            Describe the semantics of a given table.
            Task: Describe in one phrase the information within a table using continuos text, without itemization.
            Instructions: 
                1. Look at the input given to you.
                2. Look at the table and columns names.
                3. Look at the column values in detail. 
                4. Describe the target table. 
            """
            
    user_msg =  f"""Table Name: {table_name}
                    Columns Names: {','.join(table.columns)}
                    Table values: {table.values}
                    Description:""".strip()
    
    return system_msg, user_msg

In [ ]:
def generate_predictions(table, table_name, client):
    
    system_msg_describe_table, user_msg_describe_table = generate_prompt_table_prediction(table, table_name)
    result = execute_prompt(client, system_msg_describe_table, user_msg_describe_table)
    description = result.choices[0].message.content.split('Description: ')[-1].strip()

    description_df = pd.DataFrame({
                               "TableName": [table_name],
                               "Description":[description]
                               })
    
    return description_df

In [ ]:
@retry(wait=wait_random_exponential(min=1,max=60), stop=stop_after_attempt(6))
def execute_prompt(client, system_msg, user_msg):
    completion = client.chat.completions.create(
                                            model="gpt-4o",
                                            messages=[
                                                        {
                                                            "role": "system", 
                                                             "content": f"{system_msg}"
                                                        },
                                                        {
                                                            "role": "user",
                                                            "content": f"{user_msg}"
                                                        }
                                                    ]
                                            )
    return completion

In [ ]:
client = OpenAI()

In [ ]:
BENCHMARK = 'NextiaJD-S' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider
filenames = glob.glob(f'{BENCHMARK}/datasets/*.csv')

In [ ]:
filenames

In [ ]:
descriptions_df_complete = pd.DataFrame((), columns=['TableName','Description'])

for filename in tqdm(filenames):
    
    df = pd.read_csv(filename)
    table_name = filename.split('\\')[-1]
    descriptions_df = generate_predictions(df, table_name, client)
    descriptions_df_complete = pd.concat([descriptions_df_complete, descriptions_df])
    descriptions_df_complete.reset_index(drop=True)
    descriptions_df_complete.to_csv(f'{BENCHMARK}/table_descriptions.csv', index=False)